In [1]:
import pandas as pd
import numpy as np
import torch
from torch import nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import copy
import time
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import r2_score
scaler = MinMaxScaler()

csv_file = '../data/Climate/DailyDelhiClimateTrain.csv'
df = pd.read_csv(csv_file)

device = torch.device('cuda')

In [2]:
print(df)

            date   meantemp    humidity  wind_speed  meanpressure
0     2013-01-01  10.000000   84.500000    0.000000   1015.666667
1     2013-01-02   7.400000   92.000000    2.980000   1017.800000
2     2013-01-03   7.166667   87.000000    4.633333   1018.666667
3     2013-01-04   8.666667   71.333333    1.233333   1017.166667
4     2013-01-05   6.000000   86.833333    3.700000   1016.500000
...          ...        ...         ...         ...           ...
1457  2016-12-28  17.217391   68.043478    3.547826   1015.565217
1458  2016-12-29  15.238095   87.857143    6.000000   1016.904762
1459  2016-12-30  14.095238   89.666667    6.266667   1017.904762
1460  2016-12-31  15.052632   87.000000    7.325000   1016.100000
1461  2017-01-01  10.000000  100.000000    0.000000   1016.000000

[1462 rows x 5 columns]


In [3]:
df['date'] = pd.to_datetime(df['date'])
df['dayofyear'] = df['date'].dt.dayofyear

df['dayofyear'] = df['dayofyear'] / 365.0

x_columns = ['dayofyear', 'humidity', 'wind_speed', 'meanpressure']
y_column = 'meantemp'
x = df[x_columns].to_numpy()
x = scaler.fit_transform(x)
y = df[y_column].to_numpy()

print("x shape:", x.shape)
print("y shape:", y.shape)

x shape: (1462, 4)
y shape: (1462,)


In [4]:
class MLP(nn.Module):
    def __init__(self, input, num_classes):
        super().__init__()
        self.num_classes = num_classes
        self.ln1 = nn.Linear(input, 1024)
        self.bn1 = nn.BatchNorm1d(1024)
        self.ln2 = nn.Linear(1024, 512)
        self.bn2 = nn.BatchNorm1d(512)
        self.ln3 = nn.Linear(512, 256)
        self.bn3 = nn.BatchNorm1d(256)
        self.relu = nn.ReLU()
        self.linear = nn.Linear(256, self.num_classes)
        
    def forward(self, x):
        x = self.relu(self.bn1(self.ln1(x)))
        x = self.relu(self.bn2(self.ln2(x)))
        x = self.relu(self.bn3(self.ln3(x)))
        out = self.linear(x)
        return out


In [5]:
class CustomDataset(Dataset):
    def __init__(self, data, labels, device):
        self.device = device
        self.data = data
        self.labels = labels

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        return self.data[idx], self.labels[idx]


In [6]:
class Optim:
    def __init__(self, device, lr, num_classes, epochs, batch_size):
        self.device = device

        self.lr = lr
        self.num_classes = num_classes
        self.epochs = epochs
        self.batch_size = batch_size

        self.model = MLP(x.shape[1], 1)
        self.model = self.model.to(self.device)
        # summary(self.model, input_size=(x.shape[1]))
        # exit()
    def Train(self, data):
        def N2T(data): 
            return torch.tensor(data, device=self.device, dtype=torch.float32)
        def D2H(data): return torch.nn.functional.one_hot(torch.tensor(data, device=self.device), num_classes=self.num_classes)
        
        Tx, Ty, Vx, Vy, TTx, TTy = data
        # ====================
        dataset = CustomDataset(N2T(Tx), N2T(Ty), device=self.device)
        dataloader = DataLoader(dataset, batch_size=self.batch_size, shuffle=True)
        # ====================
        optimizer = torch.optim.Adam(self.model.parameters(), lr=self.lr, amsgrad='amsgrad')
        self.model.train()
        best_model = copy.deepcopy(self.model)
        history = []
        for epoch in range(self.epochs):
            epoch_start_time = time.time()
            losses = []
            for x, y in dataloader:
                optimizer.zero_grad()
                loss = self.Loss(x, y, self.model)
                loss.backward()
                optimizer.step()
                losses.append(loss.item())
            losses = np.array(losses)
            avg_loss = np.mean(losses)
            
            loss_valid = self.Loss(N2T(Vx), N2T(Vy), self.model)
            loss_valid_comparison = self.Loss(N2T(Vx), N2T(Vy), best_model)

            epoch_train_time = time.time() - epoch_start_time
            print('Epoch : {} / {} Time : {:.3f} Loss : {:.5f} loss_valid : {:.5f}'.format(epoch + 1, self.epochs, epoch_train_time, avg_loss, loss_valid))
            # loss evolution
            loss_save = [avg_loss, loss_valid.item()]
            history.append(loss_save)
        return best_model, history
    def Loss(self, x, y, model):
        y_hat = model(x).squeeze()
        loss = F.mse_loss(y_hat.float(), y.float())
        return loss
    def R2Score(self, data):
        def N2T(data): 
            return torch.tensor(data, device=self.device, dtype=torch.float32)

        Tx, Ty, Vx, Vy, TTx, TTy = data
        TTy_hat = self.model(N2T(TTx)).cpu().squeeze().detach().numpy()
        r_squared = r2_score(TTy, TTy_hat)
        print("R-squared:", r_squared)

In [7]:
from sklearn.model_selection import train_test_split


x_train, x_temp, y_train, y_temp = train_test_split(x, y, test_size=0.2, random_state=42)
x_valid, x_test, y_valid, y_test = train_test_split(x, y, test_size=0.5, random_state=42)
data = [x_train, y_train, x_valid, y_valid, x_test, y_test]

In [8]:
Trainer = Optim(device, 0.0001, 1, 200, 64)

In [9]:
model_name = 'time series'

model, history = Trainer.Train(data)


Epoch : 1 / 200 Time : 0.221 Loss : 685.46688 loss_valid : 653.41028
Epoch : 2 / 200 Time : 0.054 Loss : 650.94000 loss_valid : 633.51556
Epoch : 3 / 200 Time : 0.056 Loss : 633.83934 loss_valid : 617.66766
Epoch : 4 / 200 Time : 0.055 Loss : 626.01889 loss_valid : 604.58423
Epoch : 5 / 200 Time : 0.049 Loss : 610.48884 loss_valid : 592.59167
Epoch : 6 / 200 Time : 0.048 Loss : 601.34809 loss_valid : 580.93665
Epoch : 7 / 200 Time : 0.042 Loss : 585.64435 loss_valid : 570.05438
Epoch : 8 / 200 Time : 0.042 Loss : 580.37719 loss_valid : 559.20508
Epoch : 9 / 200 Time : 0.042 Loss : 565.58889 loss_valid : 548.57727
Epoch : 10 / 200 Time : 0.059 Loss : 554.73815 loss_valid : 538.02069
Epoch : 11 / 200 Time : 0.063 Loss : 545.24394 loss_valid : 527.74396
Epoch : 12 / 200 Time : 0.056 Loss : 533.68099 loss_valid : 517.10272
Epoch : 13 / 200 Time : 0.059 Loss : 524.70395 loss_valid : 507.17380
Epoch : 14 / 200 Time : 0.068 Loss : 512.09327 loss_valid : 496.92239
Epoch : 15 / 200 Time : 0.060

In [10]:
Trainer.R2Score(data)

R-squared: 0.9385001355076723
